# 🚀 TASK 6.4 & TASK 6.5: MA TRẬN ABLATION STUDY VÀ HIỆU CHUẨN GATE 4 ĐIỀU KIỆN
### Đánh giá đối chứng 5 cấu hình M1–M5, hiệu chuẩn nhiệt độ và tuyển chọn mô hình đại diện (100% Offline)

> **Mã nhiệm vụ**: `Task 6.4` & `Task 6.5` *(Milestone 6 - Tuần 6)*  
> **Người chủ trì**: **Thành viên C (Training & Evaluation Lead)**  
> **Người phối hợp**: **Thành viên B (Architecture Lead)**, **Thành viên A (Data & Infra Lead)**  
> **Môi trường thực thi**: Kaggle Server GPU (NVIDIA T4 / P100 / RTX PRO 6000 Blackwell, Internet: Disabled)  
> **Thời gian chạy dự kiến**: ~3 đến 5 phút (Inference-only, không huấn luyện lại)  

---
### 📌 NGUYÊN TẮC KHOA HỌC:
1. **Bảo vệ tuyệt đối 18 tập test Task 5.1**: Chỉ sử dụng tập validation đa miền độc lập (540 mẫu). 18 tập test benchmark hoàn toàn cách ly, không rò rỉ dữ liệu.
2. **Phân tách hai phân vùng độc lập**: 50% `val_calibration` (tối ưu nhiệt độ $T$ và quét ngưỡng $T^*$) và 50% `val_selection` (chấm điểm xếp hạng khách quan).
3. **Kiểm định Cổng Gate 4 điều kiện**: Đối chiếu tương đối với mốc sàn baseline gốc đã xác lập tại Task 6.2.

## 📦 BƯỚC 1: THIẾT LẬP MÃ NGUỒN VÀ BUNG TỆP ĐÁNH GIÁ (100% OFFLINE)
> Giải nén mã nguồn nhẹ, tự động bung công cụ đánh giá và manifest chuẩn 540 mẫu val mà không cần Internet.

In [ ]:
# ==============================================================================
# BƯỚC 1: THIẾT LẬP MÃ NGUỒN VÀ BUNG CÁC TỆP ĐÁNH GIÁ (100% OFFLINE KHÔNG CẦN MẠNG)
# ==============================================================================
import os
import sys
import glob
import json
import shutil
import zipfile
import base64
import zlib

print('=' * 80)
print('BƯỚC 1: ĐỒNG BỘ MÃ NGUỒN FATFORMER-XLA TRÊN KAGGLE (OFFLINE 100%)')
print('=' * 80)

WORK_DIR = '/kaggle/working/fatformer-xla'
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
os.makedirs(WORK_DIR, exist_ok=True)

# Bộ lọc bỏ qua các file checkpoint nặng tránh tràn ổ đĩa 20GB của Kaggle
def ignore_heavy(dir_path, filenames):
    ignored = []
    for f in filenames:
        if f.endswith(('.pth', '.pt', '.ckpt', '.bin', '.tar', '.zip')):
            ignored.append(f)
        if f in ['checkpoints', 'pretrained', 'test_degraded']:
            ignored.append(f)
    return ignored

# 1.1. Tìm kiếm và sao chép mã nguồn nhẹ từ Kaggle Input
zip_cands = glob.glob('/kaggle/input/**/fatformer_code.zip', recursive=True) + \
            glob.glob('/kaggle/input/**/fatformer-code*.zip', recursive=True)
train_cands = glob.glob('/kaggle/input/**/train_crossdomain.py', recursive=True)

if zip_cands:
    print(f'[*] Giải nén mã nguồn từ gói zip: {zip_cands[0]}')
    with zipfile.ZipFile(zip_cands[0], 'r') as zf:
        zf.extractall(WORK_DIR)
elif train_cands:
    root_cand = os.path.abspath(os.path.join(os.path.dirname(train_cands[0]), '..', '..'))
    print(f'[*] Sao chép mã nguồn (loại trừ file nặng) từ: {root_cand}')
    shutil.copytree(root_cand, WORK_DIR, dirs_exist_ok=True, ignore=ignore_heavy)
else:
    print('[*] Khởi tạo thư mục mã nguồn.')

# 1.2. Tự động bung tệp công cụ tools/eval_crossdomain_ablation.py offline
os.makedirs(os.path.join(WORK_DIR, 'tools'), exist_ok=True)
script_target = os.path.join(WORK_DIR, 'tools', 'eval_crossdomain_ablation.py')
b64_script_data = "eJzdfd1y20rS2L2eohenfAzKFERSlO1lhU7Jkn+0x5J9bPpkt7Qq1AgckliBAA4wkKW1dfE9Qiq5SVXu8gC5z97mOu+weZJU9/wCBCnJzlalogsbxPT09Ez39PT09DQ8z9t6zcTrrFjyYufP7w5GcJilczis4IilC3gTMzhhMClYCgcXCRNxlsInUU1v4Gd4G/MKDhcVS+ENExyGcIRvfol5ujWJowUsshySmKewyFgKE1ZewtNgH3yD6YSJIr6Gk/7OyX4HrpiGGYJ/yJL4opBgP8NJNuUJfOIJj/BNJ9ja+rW6AVHE6QLEoopALOLRFgBAP4BTlsM+RKyCBZYvM/l/lc7hKuaphAOAHTjpj+AlK3kSpxwEi2AeMxjuRAkrS/BnTMxoZMIhvQmjy1wEuVh0HAyDEXxIcHjgdZxyUaXcrZgnLA1ZeMFLEX7h8XwhyiaCvREcMcHgfZrcgL/c242KrCyn2ZLFabjEjlP1ZrXhCI7iUsRJAk+QaSX4y+E96+6P4F328QB+iydwMGW54AX4y/07aw8CmBTI2euKCShRUuifJJvHogRR8BT2hz1A0bliSejggwwiBgMQBUvnIBZMsYsIOkw4S0dwkC5gnkXwe8VAoNxcV5DcQEQidvju+IOtccTnBZvyKfw63uvJmilP4U8fXr2BZRVBtGACkgqJ2+tRtb0AJiQtKItrZNrXMtpxiDvKYMoiOGFRkcHB4WEXPnKWyKfX7JLLJ1X8+bALBx/kSERMdgwFG8m0KF8iIUgdlBWUlUCxW8ICZ0+Kw3PFYIqkz7McIhwMFN1lNq0SDpc3KPBMWGx/RmbM4oTD4affsO6fPr0/BYEIFllOcMPAna1/ZhF8wko/42RdwhGS05zCZjCG7mB8WLAUokWMBEaQsHwkGe3MVn+/94gmMxaUesrK1xbRJIuhqlAzQLqIuYBpBhO4wN6/23n5+s0nxPB7xQWkc2JjxARMtuXINpp0eIWSN7fUWXBLiFjwTPd2ir29RIVlCbMCTmMqCinmUCCfshTUNA6zNLnBqQHRIjM6ZlplEYjqBtm/yNItz/O2tuJlnhUCslI/lTfmMc700zzJLvTz38os1c8iXnL9zIp5zoqSb82KbAniJuclqKJP8TJP+Clb8jJnkYWI07kGOYoj0YV3cSm6MKnyhHfhfY5jwpIuHKQ3htK0WuY3wEpIc4nnw/E7jeR4yebcQIqsiBa1H0GaUs20+TaYVSlxgCUI8FpRSKWViJMymKIa1LQywUouuvTwLmNTXtQQXsVlnKUBqpMSdW2JOO2vra0PH9//6dXhJPz4/v0ExpCVQc7EImAXJf7v699/y+LU/JjGRcqW3A9DZH0YdrrgBYHX6WzFM6ghTDMBcYqMpJpyhuhfQZyWvBB+r1ur1NlCNAtWMiEKH4FLMc0q0QWv4FGWzuJ5VXBPTTdbHjilPk+jbBqn87FXidnOc6+ztbU15TOYxek0nMVFKUJ+jetCOveRljCJSzEipp+VojjvwM4Lw3V6I5vzPG8SL1F+0zlMWYr6AwSLYcoqqYzlRJjiDCtZZJdT8BcZFqK+QxmGnAnBi7QToPAj7llWQI7DZQnS8w3iGY1lbt/gX5SlIk4r7oJ52x7hgKwA79/L53qtJRPRgpcwJjoC/MfPu1DwqCrK+IqPJ0XFjQ7SeFWtOir8K7ioinRFchT8We/couJJPDOANP6ln1u9uQldLrGo0tMs5YqjUbbMC16W4d9yPg9/3+v5MU69kZyBAf1L3HR+G16e8pRW4RIV6pQ0Ez7R+niVxe76iEuTVLwXGUz5Ev9LFxl8PDgxHLyoZjNewBjiLHh5I3h5/N6XdBNNQZSlVyjx3sc3L71OULIr7ss6XeT+komxh217XVzck1jcjPd6EoEEC0rOL/1ebSxkv7KcpwpXp9HO1taWtNR+Y8khmhpHZGp8YHHBp0qB+Op/xQzdIfVWijMtGAyWaBuiXQBXXK6/YpHFNIxmHccfF7G1NXDwghpmZF0YxmkswtAveTIjAcyKaalmIWphnHikcs/Pu1ZrjZH7jtRgbZr8xRSFWj3Vi01lGFtEOEWsLgwOs2Weldw/q8mjA/CRl/HfuT8YDLsQp4IXeSbtorEDdOwWoEEevDw+/Pzy+LDTXYf3kGOdwyLLEfd6uEk24WmZFf56kFOUogSpXJmnS87S8VkvGD7vD/eHT592oRcM9589Hzzbp8fe80G/92zvvI6bRlBMseLg6fOngz/uDxF68LS/1xvsP6fnZ/vPnj3r989rFR0azztbDs8TniqW07SMU0fNKZFOeOq7XK3Vn3MRC740YhNPr0eIhbDRen0mV0s5WF2o/4pT0QWr0GWrEYxrYnQWT69td2iVQy0kpSuYc+F75qXXBc+zOi5hFzxBFZAK3wDTS68LvY4FnPOUF0xkqC8MoHmJWKv0Ms2+pDiDLatv6uoyXs7DiKbd2NUEhroVZaAr8uuI5wJe0X9xZrd79PcTvGZJcsGiS1iyCvT0R1sx5ZU0+ZIsvoOUlH+RzXZBzhoSb4iyJCvGfn/wvAv6n47TSUT0+14Pxq3KXTXiDgqx1rRdn/BujUYF2UYL+O+odZtC6bbSdVB0Jcu7lqFG4R5Vy+XNSxZdXmQp99M0OKHNiVWyJ8oe1mOb4x79qoJogaoVTe4lbmPkbmyBNjMa6ocfPgcb9GhaLUNqn5c0MWAMA1ddVjkv/E5gatlxoaHIswzlN00D2vPGV/zgav4hy5LB1Pf7Xeh3GhVmOHvSNHgXp5wV/l6NAmfqzrLiCyumisrrUW1i0ux1X1iCZ5xhHwxx/nUwSzIm/E4nmCVoRaV+v9NklyLNx9p2CSQPyX+IxWLClzlyC43FFsbQ//IlRBnyJ7UbMN+pDJ9wh5XOO2rDVGXABbAkAP/48OQdDHr9Zx259Km9nFiwJW6UnP3cC+jBtLpZ3Y65G0ayPWhPK8VB7v/uWFUfwHfhdIr4+YEVbMkFL3zJlyzlpd/vwDb0g/21fJVOjnszF1vFNZkKo4Qtc79JTReWcTruBf0uLNn1uN8LeivcVq6VXcLnkBaLDWR1QbBizldfL9l1GAte6Omz36MekNRZ0tMEXTcIh+yhISPz6lUqiiy/eZeVpTPGWS7iZfx3Mg/VeOKb4B1y0T9rdvq8C0kx7gW9viVnrB8c5YS95CgqCTVX1+KmzeDvvMhC9AU5FBHnI5bwaaiGT00yzVD5tl4Bm8G+un33a1jMqK5WDHBBIdT1MsPEsrQ9s8SXgue+6eQK76UyeKgEBWhE+LjsmG1EJXjIr3MeCT51p17IiyIr/LzILu4rQWl4EadG/fb3m/JD+0jtZitZjAqBHFvSi3d5A1doWb86fAX+K0USuM7eV0iS3TvSznfK04iXXcgLPo3Jh4CskoQt2bXsQBem8XKs9CWLoqpgUUy7Qb9Wb2zYqLWtZM1FnIYXWZVOWSGrSfxJnJJDBXfzfd1/eAKqIR6hUukF0rsnMoG+JoauGEShOCjbC8jIRkvJ7Ipj3MWiR5T7ErEj50hPkn2heVWn7Sy2JhyWVHneCoVUnjvWBzWB4+EMKrywDXXg53rZvxtb/FY68yLLQ4NMPuihDNAY97UMOm3PatVeQK8+oRW/bixey8Ez+e68gbtW/WoeWsItEqcz98CCvHwyhjTHzbnfjnOnSSouG74ahLJaGsyosl1pUINRm9o84maasiSqEiZ4yKooZLmelGkeID8LduNMSfvS2RsQyq7E7Dp1rM8bPdQf3x/uHHw+xL3swRUv2JzDh4JH5E2DywXOzRztsgU6MsvLhLMitY4cdslDIgzGyM+L8mzURRkjtVZMSQpx+Ip5mRXCt/Cds9FoRwFiEZ+Gqjc40eTTGWFQApuGeUb6OKdRNcBjPfHSMOXzVgBt4qJrSWIZQw+3xKrK2BU+xY/9XtDr0r+y7k/ysABHisyT3/G0iUXSqBULls7lfM8VjVG1RCqaXTPUzu4EVEu/yGkdzXHJJ+pVdXw7U29TPpdUioLl4QwFfc4FeRXTvAsevuZ/z+Kp110t+LvXJS+TMnZZhUauQuSLHB02eUG2UG9lNJoCQ4W5/kWsJBJ9/O8J0qsayS2jHPBtaB0wnDqo1an7aJbXianNIBzRJM59VkW4Y0cdjZCdTrcJkNfL1bQrv3Ceh7QisyQUi4KXiyyZSidD27oox711bZRFi3AZpyPZOq0Me/u2hF27JU9tCVoCblFv0N26c27/2jgeyfFwJslYvO6gBKYcIqYsbqkPXvfN7Da9V6LK5MIke9RV9OOawnf2u5poyWA7gET9fq/2dokthbM+jGGnr9m4TpcEdPjgd3D6xmWcloKlEddLfN345knJV/SQYk6Ykvmtlt52pKq0Fa0qs2u1WOBibQfJ6hA0L8jOcPr0Ygzoo2AlHtL46MWxi+FP8gRRuzvltA8LfGemCdksSjHgsuz2Ct/ZqrOHVXV3ubN0bdX+Ha3iNNaVNfFy2upfOP/pCaew65K6s1q6Wm3W17X8AWw7rW8bjFpt2LInpqyLQvu8xgI6unW31yIPkX/3HYzaOD6sao176dqqd3GPOqoqa+LNWNIvYgE+rbDgrmqSqiYLVC3LAnohWYCPNRYoVPpxhQWOVugF+7gYaCY/0W3ZxumcRsG/WFEsLZvDutrRj61wpLSUpb6om2kWpLuCVq0guH2r0G5zNhkP2lEZjeLqf7Uxt2cFDdVvDrojhsEOtM2y6j/KUMVLx41eHxYxnQPdacv9P6l/12pYNXR1RYs1oqwoeCRqdoecVpY+ZXGhlQ5j8s03yySvWYQ2kq9RKsNeWyXUKfkG3V1EvGOskMSUl4igqQUsH9oh+loYNQVuV84M6nOlLHZ1V01Jp0bjammNXiKmtSVDIrXUd1oyJa0tOaWrI0NMlEJCj98nd9qG7Uobs30XFaLx67LWneJfjU7wDLc9NRV988Y59PFoAF2ooglBHXchZk0IVtlCVtWLcluSq4JbpWyueBHPbsI59lB5qZi0VSOWTuMpvl9yUcRROVpRHhLVhQp32wTnWJ4XWZbImBEKFrAaiP6n8CFRMDeUhvwZcEH7xwwPoBcgZFRBFtPZ8xJLyAqVlFjtvSPDjwhLf6ROXU3gFc54DNZT5yE76K2FPG+tPdC1dYSWqUy80RgGQW8dhr0R/LrXg/e450mSGorfbSxXrcpQVmlvEs9kngB66CDPKSRsr2ciyRD0aS/oPaqNbrTg0SWqvTOpWVmShDk6+8eAkQxSjKN+OI1nMxL+hgicedRPtWKhPJ7DzooAtEBpxKox3cQLst/3HdoCluc8nfozz2Ec+HLonSEwzOqM4OvjDwefPj3Gea6boPn8+PXB8bvHt+Af8USwEXxVzY6eBIPZLdZVh3wqZERVttLjDM9rlpR6fAZ3jo+ZrpuGxwJptHp0BnZ0BkqPrh+dgR6dppQYaWwM0WDjEA02D9HgfkO0t2GIft/r3SlADRiNVA+QQv/C+knXj88e+Dgx3IH5da/XGJO9jWOyt3lM9u43JkPJ8c0Dc4fk1EE0XtWYX2vixZh0QwdYOgV/TXNm5TnXCqPTMpzWYnfGdSjHdUXqUCWhQvpZ6iJC+6gx3MOW4facVnzCasbf6ZZhggyaxdKVnj12e/b4fIQ1HnUk/jrnhvfinFrXdWFXjY5aQS+qOMEzEEMFnpvKIbsoWBotQowAHGEghVwvMeabAg7cd0mcN99N+VUccW3cy1+yJC7DKZ6XjwAXU0MsLbLmhNasq79gzJFgdEaOQXd4bs4gZbnsR57hyYs8TMHAUwoNThe4yp70KbbbWPhkSMmmyfuZCfAN5SRnjYg1U9hxjiHoyBvGMER8zhChKeid9D0pFoOmQ7URIWDPzcf03AlE5stB0schGA1aFlFAHDFhrZJf9K4OJgoWp3E6D+ywhEuWsjkvdN1DU3IiC2RLMrIdfdQwbobN2slzoWgfexh5Pvotnuy82+0PPWupmeEamydb6Pb4jqGzlaqSh2WxHPsu8B8kcKcONmcY4nkXZFksw6wSIZ2M9QZDWaL2NnSnYuyOsG+HRnHlJzjlFSSM7gy8+ef/+O8pXPzzH/8F/ud/jP/5j3+rYPHPf/y33Fwm0GK30td9z4l6qDM6SLKChUzdQlCc+y2eIE51N8FUvYqFchU7Lm9C0wWPIhFDio/lGGXULL+Ky4phrBK9cPwoREA0w8OErx5P2UXCpx4GAlYcQ3NZeumN4HkXPJbkC+aNoP8UHckesjhhN7wovREMbg2+OvG+JRoDhDCOd6xbtAcVRsu0zktd6EzLvMA4rJkHcLZ9TjdeHPXw1WHALYgK9a7GceuESq0EXRFUdvE3e/qZsaltH0+Z8zDJIhkc6EV55XVrAfFj0m0rwViTm5zTwe4PN+fEdgiM4R8bJDLOTEowlXndtjLnbWcV2Vf/8uzZ6BxZchmUghWi/BKLBdWtEh54avN52RnBFbmFL7twRaHgiIHO/kq/Y4WhoHNgapl6J0kL0VPk0yPF7MWRaI7cMi7LOMUjyIqO2tE9oUW6wMNwT0Nc8pvS68LZudOfKrXn/WsROEBtOBwR+9//9T+dw9HKMlSTsxGcSIIAkeFY1rpwC9/gs2nQwDQJdcUT9ZPmFQ4TEzIgHDnQOhP+cA6HFKHLshH8QoeZIsZAM3ZTXz2z5hRhMTx25sjjAA6xEm5XzQWQBS3IiywOdOCiVqGSvc6KZgoDdA+qY2a1Msq1TEXty4AUddlKhplIhTWypsFKSFl3xVRZE4fked5r2QLdqdKXVK4YGhB4R4W25BwP+Vlxo2ONBugs5BWcockm7zidG6OiYF9sUI1cNK6Nlda2xqE+s5VkDMROv4PlQ8vEn/BqkqZIVOYOHsWvD+QPF/pdlkNvREZlFw5ZBEmWY3jGoIsbdyTZkQ90AQ3R7grKbCbQQW0JklEjO46f2xynqIro2uyNnCBg4+yXAy79ZAa0PzrXOLtwyXmOz/WLB0j+PBbKLVKps2S5FM1jYc5HlEqc++ZoRLrQ69A1YiS0cr3XoXW0pwwnYsI/s62pMDKqaMivSa0dMSW6RZW69/tCpq7ShSydkoPKJzNCMk3OUW/swTY8VzSpd68PJq/ffzx59VFePz05gMnHg1M4ePnuYHL8/hQ+TT4f/QX8ycGnX+hmHl48PX71GQ7ffj44hTcHk1emcKi3elolTCgS8SIewVckRlnlqIkOZRyqDk/1yVztaDiB1zXJcNbqqNkBx+QPS4GH804Dejq45WPwomrK5HxAU1zxoZqyIC5DdsXiBC0PN9JNNVpXax9uJlhTaSMWq6ANdQfn8PPRAV6qw0OBisEFE/AGg2qbWt3Dx0klrzk4ykCoEZMT73GUV4/x0BhjOEUR64tA6j4JnUC4uGsDQsu1M05GRuVP3wIbW7OPdzSXMjr4isE1i9TdFXMrCW1xY5rbvZjZz5G3zG4QkSctxrlH1vxOfxjkwjHovbzgtK3g0901EEcHk4NPrya7d0PuXrL5POG7cZpXYhevtpVclLt4FXqecH61ay4J7+AuQZQ7OYsu74ftvlWlzip4mSVXuMAmMTrJ266KNUaxgxvG+uhpHg0Cuj+q7xSpy2CN1XVfbkmtc0Lvz7B9x+N+0kdfdws99asqRMmyT/ev67c42hiy9tq2M6BNdt+7zr0Bv5P/34n/+5Gdu2cPJ4MH8GOwiR+W6+Xundfhve9F8i8Z+HsS+TAG3I20zoi9BzBir40RdfK2t9dc7g+Xe3d3bnt7dxuBt5d72/eDvi/ajekO7lF7+werU6/uK4LfReyXrLiM07kjINcJ2/2/jv67EX1hsdgVrLx8urP3Q2TV5Xf4APkd/pD8Dh8iv8N7y+/90G5MuXGP2j8mv8MHyu/wXyu/P4L+uxHdLb/D75Hf/QfI7/4Pye/+Q+R3/97yez+0G5O+3KP2j8nv/gPld/9fK78/gv67Ed0tv/sPk99bHV9D+7y/pugfnpisO9Jwv4qdwxFtrdsESWpLh+7NC+nNiVM4Q2u9SzYi/rtH/w7p333vvM0ftwNfZfVbdEDXtgFnsuAc9xmPzzAtjfXU7Ujfm74qqs7Dzh/jXlxtQvZkkqclS+MZLwVMK0w0VWG8M25gdGi6Lt+4P9RALbs8XVSSGRnqnwGmZ3F3Z5tL7yeJ92irsZ0zvW/f0rV0Xo0fRoXbtAIquCSetWDWzoqGV2S0/gQBveVAN7VXsOFhiteFlQwmmLlltpp+YxlSPpgxJcORJwOz5u063QcJK7386q30ZtfgG/0uSMALuoKl3uIgSCxlnsTCIzeld8USdXrflHHpGEdRxDshCzzfmcNX9LQ7bXVu6aq5lU15IrMyPrcbLrHjGDk+7xoNfziHd1lMd2wN4/wW9Oje4s4kwpM9UyGqlCMpyjS1XXTEyjwYLKaA0r2nPZncS1TwJhZvqwstOs0uyzs3d4qP9nBtn8OJpqTh2LL0BJjeaW6IivDKPqYxQcpMP1ZJDDCDz3qBVSeOVZEk8UVQ8N8rXoq6mLEvYVWgR9ZbCJGXo93dgn0J5rFYVBdVyQvMVsNTEUTZcvcd57+9xa2muNxrzHTagd5vvtfax2wq4VRO9NUpitlwHo6z3t2gKhL0hcT8ipNvvCqSrm23s2aGG4D/b2f2EbNiZkbcTvXR3XP9D98/qdUB1oKv0tCYydqRW3D0sFlvKTPZETGPGU/hF1qJKNNFxOQaSZE3pWBzPKmbxkU9fdKq0YZhPRXe39KVvPYMS7SeNFGjQlhRFbXLoBhEFcpsN43KZz3LKeqoOdm00ZiG7U4DjcsBRTwP0QtZbM52orqA2qtxGi8xyEAtb3U0sHUJsir4ZXVBbcvygCTRb8HROdvpnwcJHgznvrf71782CMO/lH+RuNxEZnL0urKh1TqrSaoIS+NyvRnkM2d0zjGSHoHbQR1+PBmDvWdBUmDLVu791qeblmOU22tWW1MdLLcNCZ9i2qavsuuN5S1CQ1JGQNGhiFlSVmbGNGZabBurFWrY1uWtdRkz4Vv9Xo9UgT5imnKb9SSPc4oUrC9NrWZZy03xfq/X4JhTUwcBWie7/nOYOYKZR30Lv8a3QY5zeBVcpvYZQQyP3BglU24z+ozgzCunV/19nEPLePq3rCpSfoO/SnGT8DlL8TkvMvU0ZUnCvfMzRL3fkpdJ6eiRVNC14lvD32FgB1un8/rZTReIYMr9C+ONOcJc1lI1XJPoMrNF5ytUXTxUixYhHmSPSVTs7y6Ui2o2S7gM5pD5YtDs50UpYZ0XpiP7gc4xeoUyT4lUMKaKDtnxNCVisK8P7+t7Ow+eQOup6pvjg2M4en9winH0k7efYfL24AO8e//mePIJJvj+6PjVKUw+4j8HH+CX41cncHR8+haODuDk+NXppnNPGbilF/Gvt3azSOf/37NZlNtUsu9MSja5KT1TkRq35/XZUg9faw3o1H9OWMJYPtdFzsR9jJubVAJuCKgN+6sdrNWB5OHm2A0D1X86JnPc0DMGyLkjJ0PP5bk7JU6sKwaMnF1bqC+8rJaYmesW2nqYOwqTjgb4j5MigQw/lc1T51yp6yGUgWudSOpa5pAyd7zmPC1RNOTcWl1yVEUY66dmQE0dWOa4ov9XYkndP4dqtswDVoksYqVQFQK8NYaWK0X9jd239szeDYd1/xJD8fo4nq4dEClNnTWoZH82I6IB1WhW8KwIi14LFJ1BlFe+SXqySkdDmmxtHOI76rrCpiu2Zstpkb+AXwuERwFxusUTlpd8WhdG2AFhzT2M8bZMsJEtKyMhQ1qc+9ayphxzW68xAq21nLQVpp7be1OprqmkutQapXYKjn/qkols3Bu5PWv4Cy2NBgzFog6kCFIQegrWYSwXvFGTJQ1QxYqw5HhrTf2yMLdtvr8XgNtgEJjViyLovqp6FOBfgk9bJ4c8uXXqmOA6/PsJM9DHkJPZpvKSvvnw2ZTjEmBjw2thN405XJ/BTvANX+biJoxYtEBdp5bkpw9KFv7gNXkwgj8f/AWOPp++WRfwpJZhJ/Bp04psgq+WMqO+UffftyrjpGuR2ubSAmOYnhlZc5aQ3MxJG3I31ReplPA2wsxkNTkha5UceXeqWC1BKS5Ua+13sU1mQ3PbWF8bHtNF6zouScIaTPUVrYnHFUvBwmmR4QZNYz1z7pfidSFLea3ESc+YfVnVE5IV3khxtTFRmzf5Rutaaa3nXGx1q9lbR621nMuubi1776m1lrwA61bAN2tg8yZovgJZv4I2ah/1ljrNPlOVtT2uXehya6ztLdZw+krAbT0luLwOttpLJVj08QOa8N7IEbaNWnt6Vntx3qa+G2pEL+RF9sVoxmOTPaa4nGZfrFasK8FvcErW+zf5LYEJXmPahV9uYEJ5Hr+p27l4E00/0yWyb3QtVr4299XkI91P+wb/6z/Dp0rgurCEb/azC/Ctrh+/wWhnZ2cE8n+o/3zA/wqr2kJMeRm1RLB55hsi5tshE/kxkY7XjK7yVr4W8unjyZM3dJ2nDo0hQN4TOKrgHZ5tHTF4mcGnOHUP/WSghfdEfdnhbcbMV0F+pq+CeM1Dbe9Jy7c/TjLANVq3b3d05ExriIXjrEM/2pnWSU6ENPGgvjXwvsH29teL2+1t+Gai73E8zy7OMSL3a3H2uKG+9GXAWqmeaU5h3UMw8xC4pg8aiNxZ3FLUvIu4poWVufj4XF53rNPLKgeRxWMmFH29w/1wR21CZXg+ecnR/0nx1EFWiZwudBVdIB9emF06XteovDKR2E3n4Ep9rzV+W839qLxSsm8d/i72Lnhf7uHzX3Dc7aGpgHal32jlrHce4E0Qd0sxC75gZgPf63qSboWig3bVXzFptIG8U0Bb8ZV4AeZscd4hBGQYtTbh+Cb0YQB9FoT0X5nJE+eGUVjQDVhnmByH5LMf+gjLg43MvZEbKH/69vjVBI7ew88w+fyXV6dw+Pb9KUzevnovY+iH6BD6jH6g03Xx9A2b8yc4whPHRjIHqS3RF6vu99bV5aqB8mwvGKg8aA6Esyz/8Xmw7+xvVq2O4fOg5wI0TYH9QTBolrv4/7ha7GDfN6W3NQdx87Z328W9VsDOvU7PW6s+8JztYrnpjK3BpIulPBBptul14WLZueP0uykFMhe0vEZEh92t3fmBA299H4JupJlWTUqT2umY/WbRfu/RSkq8K/vafiMoUinKy0x9aEnnmk0rguNTySB80ukobdZX3Ni62ydcys+djZKaVFx+tIEwFyyd4gvf4JFA5AgP4ylu6WwTu7swUFdHp9eyN3hfiRfLs5GpocI6pteqXxrCAIwkhBnW7XNnqCiZ1BVGBOnc3T52diw5iY8YT+CuizOvObRjeTxqaFSHo98ag+2AyZcSTjOQkt0UvKwS2nGqja2ExINwGQ8l97hj+QGVGgAmQM7KWPCwjDJKAe4kIfxXbo9DzBuzbouMF97cYTYFapNjuLqyb8ZPOMjS85VKhtPttWSxWw2VnVvJ3W23VhGGMKeXbSQJi7cJqXE6g9EP4HUswM0/35rAUiZPsWpDl9HHJlpz4LsfR1jm4RVdrbPVAsykXhvzru5jzQ81CMBNtylUyvsNXyPTKcipiI5qrV/Dtl9vuxNMuWDRwu+sekfEAonvYm482Yl1uUvr7bZ3Z4/OnDALtygwU9MVg5K+l6gzdTtOGQwIMby0fajL3Cq9POLhBZ/JKXefLOROQ0i00gWWat2vuqRvHFSFY8OoIpVshpugexHZQkMrrXQ8eZ+v0dHHmFC7Trow2XYupMuGavNzTVctTFtHWzxldD1znZNrYw9dn5eUx1Xv2X2wW5I343bmeTpttSjvdHsROd/p+1J1v8cBpqpu9Au1+avuJHeN0+ouUtd4rjaQeetKs0n0tu7DiXKZlgl31IpNcXXj9px1Lj+7NVO0NomOsFmzgNujYfuFxZFNaDyPK/3FTXQgqRRrB4eHVooMJp1zdJOcwBMXqJU3llbqXpzOsoe6bI3+tmuWNzJr1TpgPVMQVK4LDd+fUb7IbPOjBYqUnwKi5waM3gbQQFueNSZC3cZCYP2meeqkxQQPnYzItMBI+dFQ8lebz9I1D7XD0jCj/gUC0x7t1awwvFhrLNY3IBtMSvOmvULDSJUPTZeq/mIrzbF17tRTuqB9hB/X+Qan0hiZoEsN1/IJreU7L+ATmdl2Lvi/YX5kx6WqX/zCBfxaqWm90YX6cJ8pGtdztKtdFtWdh/MW5yHm+ahQyXvb25iDbHsb/Al+J3YIR790PGLkmSNH5zIji4eJybwHuCDnZ49b5h766vZmt00APd+UK6/NJTg/e2wnGsINZ7fIC11Ak0u/b6+vJtfj842u0DrgOmemHMbbVp8jcj1BJ5TzpVon4LFVbN3sG+YTalhTReJEmTOj5LSIGCUlkN6yk72dE5k6Ev8oQLZxLWPeLjGK4VpO1EatsTmz4hPPVpCP7jUp0a5q1sQMHjfjhC0vpgyzsFyfrai6806LDNcz1GxqFanfargbd17AyXt4K6OxjmWk1tHn94eO524EZ19bUd6eg39o+PCJlBh8XavfVLK9ujf6ft89rnnD+DXq59CFuTv6P1pkJU/p8hyq0HrsVWvvamx2q7dlzbLFza9HVZZS9fXDOxzl9h636zJvjkgjStd1rMibC/bT0oJ2A40RFhVekzJU39YD3jYkzLJ1NqfM0upV5p+6Z+6senWV8wY/8GoxdVeGtF5pRtGR4fLCGWn0CGAuoJWa+P0QzFVHGaUHw86meGFnQN2g/CbOW/C/GhqkAj952ZR69CaSxtLLMJ7GEIA0PkL0pN5TXtyNI1pNZr+nDBl5I0Mtk3FK4afKXHbzUXOBR0uYcq5hUuJ9QTzL06cD8DOGiAB+onVeMMyv9eqaRxWdEDdDg+IlLwVb4lkzBVaVophRcJX36C87j5Y7j6bw6O3o0cno0Sc3lSBVRr8fBhVpH+BKsTtnvVG71ltXadWIXFfk2IGO437Fcz2qbSwcyMZ5Efao/saBdZcia43ST/dYwL33VpeWex6UkZ9+Wi1z3xWILuCHYNMpT8V4gEjKquAhK6M4dhPGrdxauW6VaLLyfrZnVnRa1aR3Xcoj593ZWzxgmhxMzmUs8cv3MJHhxLhmTd5+PoTJ22P48Pbz+9M3gMFNGH98+hYO8TfFNZ0cnOLVGJlLir5pT2kndUQpvVFpleg5OCjm1ZKn4gOV+HhyXMR0VDD2KLB7R0Z22979LPv7yfhaPqiwe91BwhSw6RSbJuS+t7NjVyqvC4xqjr1SZAUPRVGhZlzwJB97dEnVfsjUfr+SjHl9I1WH/+vvm25u28QWe13A+LUxpWWf8hmrEjGupy1SZByZuxA6LRK4YBub09eJ2lu7383UFTJagDYS0XLgtIGYJJuXu7RpdxbiJoo6aSfNwynd0EayrD6/myDT1GRRwbKK5PS75ALT7tllxrlyLSq84KyjZtZwRyY8am/eEUODFHO/bkQ4eCjCwR0I9x6KcO8OhMOHIhzegXD/oQj3Nb/Wiqu+7aGR0oe3NdL+U430JQICAhq/1WZanbshragHGvOnDBKabgqaDkFZutiMntbuNrxDg/gtEzCPEXM6ZxV9yPeOydti96/Tmro5mVdXzRdrwK0xpvP2dmWMbztbZZ49+ijLmhR7ymMg09fqmatTBSK8SqFHxrMyASkLoqLGXa+2trZi/DYx3i4JQwo4DkPSSqEKOlaZpmuVCOf9Mihu/R9+uJZV"
with open(script_target, 'w', encoding='utf-8') as f:
    f.write(zlib.decompress(base64.b64decode(b64_script_data.encode('ascii'))).decode('utf-8'))
print(f'[✓] Đã tạo thành công công cụ đánh giá offline: {script_target} ({os.path.getsize(script_target):,} bytes)')

# 1.3. Tự động bung tệp manifest chuẩn 3600 ảnh (có 540 mẫu val) offline
os.makedirs(os.path.join(WORK_DIR, 'DATASET', 'manifests'), exist_ok=True)
manifest_target = os.path.join(WORK_DIR, 'DATASET', 'manifests', 'data_manifest.json')
b64_manifest_data = "..."
with open(manifest_target, 'w', encoding='utf-8') as f:
    f.write(zlib.decompress(base64.b64decode(b64_manifest_data.encode('ascii'))).decode('utf-8'))
print(f'[✓] Đã tạo tệp data_manifest.json offline: {manifest_target} ({os.path.getsize(manifest_target):,} bytes)')

# 1.4. Tự động tạo tệp mốc sàn baseline metrics offline
os.makedirs(os.path.join(WORK_DIR, 'DATASET', 'logs'), exist_ok=True)
base_metrics_path = os.path.join(WORK_DIR, 'DATASET', 'logs', 'val_crossdomain_baseline_metrics.json')
base_metrics_data = {
    'baseline_metrics': {
        'clean_macro_acc': 73.25,
        'clean_real_acc': 98.50,
        'clean_fake_acc': 48.00,
        'q30_macro_acc': 52.20,
        'q30_real_acc': 99.20,
        'q30_fake_acc': 5.20
    }
}
with open(base_metrics_path, 'w', encoding='utf-8') as f:
    json.dump(base_metrics_data, f, indent=2)
print(f'[✓] Đã tạo mốc sàn baseline tham chiếu: {base_metrics_path}')

# 1.5. Xử lý tokenizer BPE offline
clip_dir = os.path.join(WORK_DIR, 'src', 'models', 'clip')
os.makedirs(clip_dir, exist_ok=True)
target_bpe = os.path.join(clip_dir, 'bpe_simple_vocab_16e6.txt.gz')
bpe_cands = glob.glob('/kaggle/**/bpe_simple_vocab_16e6.txt.gz', recursive=True)
if not os.path.exists(target_bpe) and bpe_cands:
    shutil.copy2(bpe_cands[0], target_bpe)
    print(f'[✓] Đã đồng bộ BPE vocab offline: {target_bpe}')

if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)
print('[✓] Hoàn tất thiết lập Bước 1: Môi trường sẵn sàng 100% Offline!')


## 🔍 BƯỚC 2: ĐỊNH VỊ CHECKPOINTS VÀ DỮ LIỆU ĐA MIỀN
> Tự động dò tìm các checkpoint M1–M5 từ Input Dataset và ánh xạ 540 đường dẫn ảnh validation.

In [ ]:
# ==============================================================================
# BƯỚC 2: ĐỊNH VỊ CÁC TÀI NGUYÊN CHECKPOINT M1–M5 VÀ KHO DỮ LIỆU ĐỐI CHỨNG
# ==============================================================================
def find_file(patterns):
    for pat in patterns:
        matches = glob.glob(pat, recursive=True)
        if matches:
            return os.path.abspath(matches[0])
    return None

print('=' * 80)
print('BƯỚC 2: ĐỊNH VỊ CHECKPOINTS VÀ DỮ LIỆU ĐA MIỀN')
print('=' * 80)

# In toàn bộ các tệp .pth trong /kaggle/input để tiện theo dõi
input_pths = glob.glob('/kaggle/input/**/*.pth', recursive=True)
print(f'[*] Tìm thấy {len(input_pths)} tệp checkpoint .pth trong /kaggle/input/:')
for ip in input_pths:
    print(f'    -> {ip}')

clip_path = find_file([
    '/kaggle/input/**/ViT-L-14.pt',
    f'{WORK_DIR}/DATASET/pretrained/ViT-L-14.pt',
    f'{WORK_DIR}/pretrained/ViT-L-14.pt',
    '/kaggle/working/**/ViT-L-14.pt'
])
print(f'\n[1] CLIP Backbone: {clip_path}')

m1_ckpt = find_file([
    '/kaggle/input/**/fatformer_4class_ckpt.pth',
    f'{WORK_DIR}/DATASET/pretrained/fatformer_4class_ckpt.pth',
    f'{WORK_DIR}/fatformer_4class_ckpt.pth'
])
print(f'[2] M1 (Baseline 4-class): {m1_ckpt}')

m2_ckpt = find_file([
    '/kaggle/input/**/fatformer_plan_a_best_weights.pth',
    '/kaggle/input/**/fatformer_plan_a*.pth',
    f'{WORK_DIR}/DATASET/checkpoints/fatformer_plan_a_best_weights.pth',
    f'{WORK_DIR}/DATASET/checkpoints/fatformer_plan_a_best.pth'
])
print(f'[3] M2 (Plan A Finetune): {m2_ckpt}')

m3_ckpt = find_file([
    '/kaggle/input/**/crossdomain_model_best_m3.pth',
    '/kaggle/input/**/*best*m3*.pth',
    '/kaggle/input/**/*_m3.pth',
    '/kaggle/input/**/m3/crossdomain_model_best.pth',
    '/kaggle/input/**/m3*/crossdomain_model_best.pth',
    '/kaggle/input/**/m3*best*.pth',
    '/kaggle/input/**/m3*.pth',
    f'{WORK_DIR}/DATASET/checkpoints/m3/crossdomain_model_best.pth',
    '/kaggle/working/checkpoints/m3/crossdomain_model_best.pth'
])
print(f'[4] M3 (Data Only): {m3_ckpt}')

m4_ckpt = find_file([
    '/kaggle/input/**/crossdomain_model_best_m4.pth',
    '/kaggle/input/**/*best*m4*.pth',
    '/kaggle/input/**/*_m4.pth',
    '/kaggle/input/**/m4/crossdomain_model_best.pth',
    '/kaggle/input/**/m4*/crossdomain_model_best.pth',
    '/kaggle/input/**/m4*best*.pth',
    '/kaggle/input/**/m4*.pth',
    f'{WORK_DIR}/DATASET/checkpoints/m4/crossdomain_model_best.pth',
    '/kaggle/working/checkpoints/m4/crossdomain_model_best.pth'
])
print(f'[5] M4 (Distill + Cons): {m4_ckpt}')

m5_ckpt = find_file([
    '/kaggle/input/**/crossdomain_model_best_m5.pth',
    '/kaggle/input/**/*best*m5*.pth',
    '/kaggle/input/**/*_m5.pth',
    '/kaggle/input/**/m5/crossdomain_model_best.pth',
    '/kaggle/input/**/m5*/crossdomain_model_best.pth',
    '/kaggle/input/**/m5*best*.pth',
    '/kaggle/input/**/m5*.pth',
    f'{WORK_DIR}/DATASET/checkpoints/m5/crossdomain_model_best.pth',
    '/kaggle/working/checkpoints/m5/crossdomain_model_best.pth'
])
print(f'[6] M5 (LoRA ViT Adapter): {m5_ckpt}')

if not (m3_ckpt and m4_ckpt and m5_ckpt):
    print('\n[!] LƯU Ý: M3, M4 hoặc M5 đang là None!')
    print('    Hãy bấm Add Input -> Chọn dataset fatformer-task-6-3-data để thêm checkpoint.')

# Định vị và ánh xạ tập dữ liệu ảnh validation
manifest_path = f'{WORK_DIR}/DATASET/manifests/data_manifest.json'
with open(manifest_path, 'r', encoding='utf-8') as f:
    m_data = json.load(f)
val_recs = [r for r in m_data.get('records', []) if r.get('split') == 'val']
print(f'\n[*] Số lượng mẫu validation trong manifest: {len(val_recs)} mẫu')

staging_candidates = glob.glob('/kaggle/input/**/diffusion_staging', recursive=True)
if staging_candidates:
    base_s = staging_candidates[0]
    print(f'[*] Phát hiện thư mục ảnh thực tế: {base_s}')
    remap_ok = 0
    for r in m_data.get('records', []):
        orig_p = r.get('file_path', '')
        if 'diffusion_staging' in orig_p:
            sub_p = orig_p.split('diffusion_staging')[-1].lstrip('/\\')
            candidates_p = [
                os.path.join(base_s, sub_p),
                os.path.join(base_s, 'diffusion_staging', sub_p),
                os.path.join(os.path.dirname(base_s), sub_p),
            ]
            for cp in candidates_p:
                if os.path.exists(cp):
                    r['file_path'] = cp
                    remap_ok += 1
                    break
    with open(manifest_path, 'w', encoding='utf-8') as f:
        json.dump(m_data, f, indent=2)
    val_exist = sum(1 for r in m_data.get('records', []) if r.get('split') == 'val' and os.path.exists(r.get('file_path', '')))
    print(f'[✓] Ánh xạ đường dẫn thành công: {val_exist}/{len(val_recs)} ảnh validation sẵn sàng trên đĩa Kaggle!')
else:
    print('[!] Cảnh báo: Không tìm thấy thư mục diffusion_staging trong /kaggle/input/')


## ⚡ BƯỚC 3: THỰC THI ĐO ĐẠC MA TRẬN ABLATION & KIỂM ĐỊNH CỔNG GATE
> Chạy suy luận 5 mô hình trên 540 ảnh Clean và Nén sâu Q=30, hiệu chuẩn nhiệt độ và kiểm tra Gate 4 điều kiện.

In [ ]:
# ==============================================================================
# BƯỚC 3: THỰC THI ĐO ĐẠC MA TRẬN ABLATION (TASK 6.5) & HIỆU CHUẨN GATE (TASK 6.4)
# ==============================================================================
import torch
import subprocess

OUTPUT_LOGS = '/kaggle/working/output_logs'
os.makedirs(OUTPUT_LOGS, exist_ok=True)

chosen_device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'[*] Thiết bị suy luận: {chosen_device} (CUDA available: {torch.cuda.is_available()})')

cmd = [
    sys.executable, '-u', f'{WORK_DIR}/tools/eval_crossdomain_ablation.py',
    '--manifest', f'{WORK_DIR}/DATASET/manifests/data_manifest.json',
    '--baseline_metrics', f'{WORK_DIR}/DATASET/logs/val_crossdomain_baseline_metrics.json',
    '--output_dir', OUTPUT_LOGS,
    '--batch_size', '16',
    '--device', chosen_device,
    '--seed', '42',
]

if clip_path:
    cmd.extend(['--clip_path', clip_path])
if m1_ckpt:
    cmd.extend(['--m1_ckpt', m1_ckpt])
if m2_ckpt:
    cmd.extend(['--m2_ckpt', m2_ckpt])
if m3_ckpt:
    cmd.extend(['--m3_ckpt', m3_ckpt])
if m4_ckpt:
    cmd.extend(['--m4_ckpt', m4_ckpt])
if m5_ckpt:
    cmd.extend(['--m5_ckpt', m5_ckpt])

print('=' * 80)
print('BƯỚC 3: TIẾN HÀNH SUY LUẬN MA TRẬN ABLATION VÀ KIỂM ĐỊNH CỔNG GATE')
print(f'[*] Lệnh thực thi: {" ".join(cmd)}')
print('=' * 80)

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end='')
process.wait()

if process.returncode != 0:
    raise RuntimeError(f'Tiến trình kết thúc với mã lỗi: {process.returncode}')
else:
    print('\n[✓] TIẾN TRÌNH ĐO ĐẠC VÀ HIỆU CHUẨN ĐÃ HOÀN TẤT THÀNH CÔNG!')


## 📊 BƯỚC 4: HIỂN THỊ KẾT QUẢ TRỰC QUAN & ĐÓNG GÓI KẾT QUẢ
> Hiển thị bảng kết quả Ablation Study, kết luận Cổng Gate và đóng gói file zip kết quả ra `/kaggle/working/`.

In [ ]:
# ==============================================================================
# BƯỚC 4: HIỂN THỊ TRỰC QUAN KẾT QUẢ VÀ ĐÓNG GÓI BÁO CÁO TẢI VỀ
# ==============================================================================
import pandas as pd
from IPython.display import display, Markdown

ablation_csv = os.path.join(OUTPUT_LOGS, 'crossdomain_ablation_matrix.csv')
report_json = os.path.join(OUTPUT_LOGS, 'calibration_and_selection_report.json')

if os.path.exists(ablation_csv):
    df_abl = pd.read_csv(ablation_csv)
    display(Markdown('### 📊 BẢNG TỔNG HỢP MA TRẬN ABLATION STUDY (TASK 6.5):'))
    display(df_abl)

if os.path.exists(report_json):
    with open(report_json, 'r', encoding='utf-8') as f:
        r_data = json.load(f)
    display(Markdown(f"### 🏆 KẾT LUẬN TUYỂN CHỌN THEO GATE 4 ĐIỀU KIỆN (TASK 6.4):"))
    display(Markdown(f"- **Mô hình đại diện được chọn**: `{r_data['metadata']['selected_model']}`"))
    display(Markdown(f"- **Điểm Composite Score đạt được**: `{r_data['metadata']['selected_composite_score']:.2f}%`"))

export_zip = '/kaggle/working/task_6.4_6.5_results.zip'
with zipfile.ZipFile(export_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUTPUT_LOGS):
        for fn in files:
            fp = os.path.join(root, fn)
            arcname = os.path.relpath(fp, OUTPUT_LOGS)
            zf.write(fp, arcname)

zip_size_mb = os.path.getsize(export_zip) / (1024 * 1024)
print(f'[✓] Đã đóng gói thành công toàn bộ kết quả ra: {export_zip} ({zip_size_mb:.2f} MB)')
print('Bạn có thể tải trực tiếp tệp task_6.4_6.5_results.zip về máy để nghiệm thu!')
